In [1]:
import os
os.environ["OLLAMA_HOST"] = "http://localhost:11400" 
print(os.environ.get("OLLAMA_HOST"))

http://localhost:11400


In [2]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import Prompts.ver_2 as prompts
import importlib
importlib.reload(prompts)
from Prompts.ver_1 import Config
# import evaluate
# importlib.reload(evaluate)

In [4]:
from Prompts.ver_2 import Config
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and real reviews. Carefully analyze the following review for signs such as:
    - Overly generic or vague language
    - Exaggerated praise or criticism
    - Repetitive or templated phrasing
    - Marketing-like wording or unnatural flow
    - Specific details vs. general statements
    Review: "{text}"
    Classify this review as either "real" or "fake" (respond with only one word):
    


In [5]:
import numpy as np
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [6]:
def evaluate_model(df, results):
    # Get true labels from the dataframe
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [10]:
def run_prompt_eval(dataset_path, prompt_template, model, text_col="text"):
    
    df_local = pd.read_csv(dataset_path)
    reviews = df_local[text_col].tolist()

    prompt = PromptTemplate(
        input_variables=["text"],
        template=prompt_template,
    )

    llm = OllamaLLM(model=model)
    chain = prompt | llm

    predictions = []
    for review in reviews:
        response = chain.invoke({"text": review})
        label = response.strip().lower()

        if "fake" in label:
            cleaned_label = "fake"
        elif "deceptive" in label:
            cleaned_label = "fake"
        elif "real" in label:
            cleaned_label = "real"
        elif "truthful" in label:
            cleaned_label = "real"
        else:
            cleaned_label = "fake"  # safety fallback
        
        cleaned_label = cleaned_label.strip('.,!?;:')

        #print(f"Raw output: {response} -> Cleaned: {cleaned_label}")
        predictions.append(cleaned_label)

    accuracy, f1, conf_matrix = evaluate_model(df_local, predictions)
    return df_local, predictions, accuracy, f1, conf_matrix

# 1. Amazon Human vs Computer
Amazon_Human_VS_ComputerFake.csv - Total rows:800 , Real:400 , Fake:400

In [11]:
# Load CSV
df_reviews = pd.read_csv("/home/bumu60du/fake_review_detection/Datasets/Amazon_Human_VS_ComputerFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,source,text,is_synthetic
0,fake,Home_and_Kitchen,amazon,Air pressure was utter crap. I kept the lid o...,1
1,fake,Home_and_Kitchen,amazon,"Not what I expected, smaller than expected, bu...",1
2,fake,Home_and_Kitchen,amazon,We like a toaster that doesn't have the long h...,1
3,fake,Home_and_Kitchen,amazon,Love these pans they work great. The only prob...,1
4,fake,Home_and_Kitchen,amazon,I love the print on the back and the finish. I...,1


## 1.1 Zero Shot Prompting

In [12]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/bumu60du/fake_review_detection/Datasets/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="gemma3:12b"
 )

In [13]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: Gemma3:12b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: Gemma3:12b
Accuracy: 0.7525

F1 Score: 0.7513671666530629

Confusion Matrix:
[[328  72]
 [126 274]]


## 1.2 One-Shot Prompting

In [14]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/bumu60du/fake_review_detection/Datasets/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="gemma3:12b"
 )

In [15]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: Gemma3:12b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: Gemma3:12b
Accuracy: 0.68125

F1 Score: 0.6549151751051747

Confusion Matrix:
[[162 238]
 [ 17 383]]


## 1.3 Few-Shot Prompting

In [16]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/bumu60du/fake_review_detection/Datasets/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="gemma3:12b"
 )

In [17]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: Gemma3:12b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: Gemma3:12b
Accuracy: 0.74125

F1 Score: 0.7384344925297459

Confusion Matrix:
[[255 145]
 [ 62 338]]


# 2. Hotel Human Real vs Human Fake
Hotel_Human_VS_HumanFake.csv - Total rows:1592 , Real:796 , Fake:796

In [18]:
# Load CSV
df_reviews = pd.read_csv("/home/bumu60du/fake_review_detection/Datasets/Hotel_Human_VS_HumanFake_relabelled.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,real,conrad,Hotel,We stayed for a one night getaway with family ...,0,TripAdvisor
1,real,hyatt,Hotel,Triple A rate with upgrade to view room was le...,0,TripAdvisor
2,real,hyatt,Hotel,This comes a little late as I'm finally catchi...,0,TripAdvisor
3,real,omni,Hotel,The Omni Chicago really delivers on all fronts...,0,TripAdvisor
4,real,hyatt,Hotel,I asked for a high floor away from the elevato...,0,TripAdvisor


## 2.1 Zero Shot Prompting

In [19]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/bumu60du/fake_review_detection/Datasets/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="gemma3:12b"
 )

In [20]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: Gemma3:12b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: Gemma3:12b
Accuracy: 0.6394472361809045

F1 Score: 0.6303311860528713

Confusion Matrix:
[[384 412]
 [162 634]]


## 2.2 One-Shot Prompting

In [21]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/bumu60du/fake_review_detection/Datasets/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="gemma3:12b"
 )

In [22]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: Gemma3:12b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: Gemma3:12b
Accuracy: 0.5489949748743719

F1 Score: 0.43936183570329906

Confusion Matrix:
[[ 85 711]
 [  7 789]]


## 2.3 Few-Shot Prompting

In [23]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/bumu60du/fake_review_detection/Datasets/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="gemma3:12b"
 )

In [24]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: Gemma3:12b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: Gemma3:12b
Accuracy: 0.6099246231155779

F1 Score: 0.5598440849563353

Confusion Matrix:
[[217 579]
 [ 42 754]]


# 3. Human Real vs CG
Hotel_HumanReal_VS_CG.csv - Total rows:1592 , Real:796 , Fake:796

In [25]:
# Load CSV
df_reviews = pd.read_csv("/home/bumu60du/fake_review_detection/Datasets/Hotel_HumanReal_VS_CG.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,real,james,Hotel,My husband and I were there for a conference a...,0,Web
1,real,palmer,Hotel,Here are my experiences: - Had three rooms res...,0,Web
2,real,omni,Hotel,My wife and I travelled to Chicago and really ...,0,TripAdvisor
3,real,knickerbocker,Hotel,The location of this hotel is excellent. The h...,0,Web
4,real,fairmont,Hotel,We recently completed our second stay at the F...,0,TripAdvisor


## 3.1 Zero Shot Prompting

In [26]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/bumu60du/fake_review_detection/Datasets/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="gemma3:12b"
 )

In [27]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: Gemma3:12b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: Gemma3:12b
Accuracy: 0.7945979899497487

F1 Score: 0.794594018738052

Confusion Matrix:
[[629 167]
 [160 636]]


## 3.2 One-Shot Prompting

In [28]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/bumu60du/fake_review_detection/Datasets/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="gemma3:12b"
 )

In [29]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: Gemma3:12b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: Gemma3:12b
Accuracy: 0.6224874371859297

F1 Score: 0.5626179691096358

Confusion Matrix:
[[201 595]
 [  6 790]]


## 3.3 Few-Shot Prompting

In [30]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/bumu60du/fake_review_detection/Datasets/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="gemma3:12b"
 )

In [31]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: Gemma3:12b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: Gemma3:12b
Accuracy: 0.8391959798994975

F1 Score: 0.8374284177959462

Confusion Matrix:
[[585 211]
 [ 45 751]]
